# F1 Radio — Exploratory Data Analysis

Four views before choosing a model:
1. **Label distribution** — class balance
2. **Confidence histograms** — how reliable are the labels?
3. **Acoustic features per class** — do pitch/energy/ZCR separate the classes?
4. **Transcript length** — does word count differ by emotion?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json, os
import numpy as np

BASE = r'C:\Users\Ritarshi Roy\OneDrive\Desktop\Projects\F1 Recordings'

labels = pd.read_csv(os.path.join(BASE, 'annotations', 'labels_clean.csv'))
text   = pd.read_csv(os.path.join(BASE, 'annotations', 'preprocessed_text_clean.csv'))[
    ['clip_id', 'word_count', 'avg_word_length', 'negation_count', 'question_count']
]

# Parse acoustic_features JSON column into separate columns
acoustic_parsed = labels['acoustic_features'].apply(
    lambda x: json.loads(x) if pd.notna(x) else {}
)
acoustic_df = pd.DataFrame(acoustic_parsed.tolist())
eda = pd.concat([labels.drop(columns=['acoustic_features']), acoustic_df], axis=1)
eda = eda.merge(text, on='clip_id', how='left')

LABEL_ORDER = ['Calm', 'Frustrated', 'High Stress']
COLORS = {
    'Calm':        '#4a90d9',
    'Frustrated':  '#e67e22',
    'High Stress': '#e74c3c',
}
PALETTE = [COLORS[l] for l in LABEL_ORDER]

print(f'Total clips: {len(eda)}')
print(eda['final_label'].value_counts())

## 1. Label Distribution

In [ ]:
counts = eda['final_label'].value_counts().reindex(LABEL_ORDER)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Label Distribution (1,306 clean clips)', fontsize=14, fontweight='bold')

bars = ax1.bar(LABEL_ORDER, counts.values, color=PALETTE, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
             str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of clips')
ax1.set_ylim(0, counts.max() * 1.15)
ax1.tick_params(axis='x', labelsize=11)
ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

wedges, texts, autotexts = ax2.pie(
    counts.values, labels=LABEL_ORDER, colors=PALETTE,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(10)

plt.tight_layout()
plt.show()

## 2. Confidence Histograms

In [ ]:
conf_cols = [
    ('confidence',          'Overall Confidence\n(fusion score)'),
    ('text_confidence',     'Text Confidence\n(NLP model)'),
    ('acoustic_confidence', 'Acoustic Confidence\n(pitch/energy heuristic)'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Confidence Score Distributions by Label', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes, conf_cols):
    for label in LABEL_ORDER:
        vals = eda[eda['final_label'] == label][col].dropna()
        ax.hist(vals, bins=25, alpha=0.55, label=label, color=COLORS[label], edgecolor='none')
    ax.axvline(0.6, color='black', linestyle='--', linewidth=1, label='Review threshold (0.6)')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(eda.groupby('final_label')[['confidence', 'text_confidence', 'acoustic_confidence']]
        .describe().round(2).loc[:, (slice(None), ['mean', 'std', 'min', 'max'])])

## 3. Acoustic Feature Distributions per Class

In [ ]:
acoustic_cols = [
    ('mean_pitch',  'Mean Pitch (Hz)'),
    ('pitch_range', 'Pitch Range (Hz)'),
    ('pitch_std',   'Pitch Std Dev'),
    ('mean_energy', 'Mean Energy'),
    ('mean_zcr',    'Mean ZCR'),
    ('duration',    'Clip Duration (s)'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Acoustic Feature Distributions by Label', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes.flat, acoustic_cols):
    data = [eda[eda['final_label'] == l][col].dropna().values for l in LABEL_ORDER]
    bp = ax.boxplot(data, patch_artist=True, labels=LABEL_ORDER,
                    medianprops={'color': 'black', 'linewidth': 2},
                    flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4})
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='x', rotation=20, labelsize=9)

plt.tight_layout()
plt.show()

print('Mean acoustic features per label:')
print(eda.groupby('final_label')[['mean_pitch', 'pitch_range', 'mean_energy', 'mean_zcr', 'duration']]
        .mean().round(3))

## 4. Transcript Length

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Transcript Length by Label', fontsize=14, fontweight='bold')

for label in LABEL_ORDER:
    vals = eda[eda['final_label'] == label]['word_count'].dropna()
    axes[0].hist(vals, bins=30, alpha=0.5, label=label, color=COLORS[label], edgecolor='none')
axes[0].set_title('Word Count Histogram')
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Clips')
axes[0].legend(fontsize=9)

data = [eda[eda['final_label'] == l]['word_count'].dropna().values for l in LABEL_ORDER]
bp = axes[1].boxplot(data, patch_artist=True, labels=LABEL_ORDER,
                     medianprops={'color': 'black', 'linewidth': 2},
                     flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4})
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_title('Word Count Boxplot')
axes[1].set_ylabel('Word count')
axes[1].tick_params(axis='x', rotation=20, labelsize=9)

data2 = [eda[eda['final_label'] == l]['avg_word_length'].dropna().values for l in LABEL_ORDER]
bp2 = axes[2].boxplot(data2, patch_artist=True, labels=LABEL_ORDER,
                      medianprops={'color': 'black', 'linewidth': 2},
                      flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4})
for patch, color in zip(bp2['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[2].set_title('Avg Word Length Boxplot')
axes[2].set_ylabel('Avg characters per word')
axes[2].tick_params(axis='x', rotation=20, labelsize=9)

plt.tight_layout()
plt.show()

print('Transcript length stats per label:')
print(eda.groupby('final_label')[['word_count', 'avg_word_length', 'negation_count', 'question_count']]
        .agg(['mean', 'median', 'std']).round(2))

## 5. Model Training Results

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

BASE = r'C:\Users\Ritarshi Roy\OneDrive\Desktop\Projects\F1 Recordings'

# --- Load all training histories ---
h_frozen  = pd.read_csv(os.path.join(BASE, 'models', 'history.csv'))
h_focal   = pd.read_csv(os.path.join(BASE, 'models', 'history_focal.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History Comparison', fontsize=13, fontweight='bold')

# Loss
axes[0].plot(h_frozen['epoch'], h_frozen['val_loss'], label='Frozen BERT',   color='#4a90d9', linewidth=2, linestyle='--')
axes[0].plot(h_focal['epoch'],  h_focal['val_loss'],  label='Focal Loss',    color='#e74c3c', linewidth=2)
axes[0].set_title('Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Val F1
axes[1].plot(h_frozen['epoch'], h_frozen['val_f1'], label='Frozen BERT',  color='#4a90d9', linewidth=2, linestyle='--')
axes[1].plot(h_focal['epoch'],  h_focal['val_f1'],  label='Focal Loss',   color='#e74c3c', linewidth=2)
axes[1].axhline(0.60, color='grey', linestyle=':', linewidth=1.5, label='Target (0.60)')
axes[1].set_title('Val Macro F1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Summary table ---
results = pd.DataFrame([
    {'Model': 'Frozen BERT (baseline)', 'Val Macro F1': 0.5583, 'Test Macro F1': 0.467, 'Accuracy': '56%',
     'Calm F1': 0.69, 'Frustrated F1': 0.35, 'High Stress F1': 0.36, 'Epochs': 17},
    {'Model': 'Last BERT layer unfrozen', 'Val Macro F1': 0.5231, 'Test Macro F1': 0.508, 'Accuracy': '57%',
     'Calm F1': 0.68, 'Frustrated F1': 0.45, 'High Stress F1': 0.40, 'Epochs': 11},
    {'Model': 'Focal Loss + LR Scheduler', 'Val Macro F1': 0.6265, 'Test Macro F1': 0.591, 'Accuracy': '67%',
     'Calm F1': 0.78, 'Frustrated F1': 0.51, 'High Stress F1': 0.49, 'Epochs': 28},
])
print(results.to_string(index=False))
print()
print('Best model : best_model_focal.pt')
print('Test accuracy : 67%  |  Test Macro F1 : 0.591')